In [16]:
import torch

# following OSS configs
B, S, H = 1, 10, 2880
intermediate_size = H
top_k_experts = 4
num_experts = 128


In [17]:
# -----------------
# Token-centric MoE
# -----------------

class MoEBlock(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.gate = torch.nn.Linear(H, num_experts)

        self.mlp1_weight = torch.nn.Parameter(torch.randn(num_experts, H, 2*intermediate_size))
        self.mlp1_bias = torch.nn.Parameter(torch.randn(num_experts, 2*intermediate_size))

        self.mlp2_weight = torch.nn.Parameter(torch.randn(num_experts, intermediate_size, H))
        self.mlp2_bias = torch.nn.Parameter(torch.randn(num_experts, H))

    def forward(self, x):
        # omitted pre-norm
        out = self.gate(x) 
        expert_weights, expert_indices = torch.topk(out, top_k_experts)
        expert_weights = torch.nn.functional.softmax(expert_weights, dim=-1)

        expert_mlp1_weight, expert_mlp1_bias = self.mlp1_weight[expert_indices, ...], self.mlp1_bias[expert_indices, ...]

        t = torch.einsum('sh,sehi->sei', x, expert_mlp1_weight) + expert_mlp1_bias

        gate, up = t[..., ::2], t[..., 1::2]
        t = torch.nn.functional.silu(gate) * up

        expert_mlp2_weight, expert_mlp2_bias = self.mlp2_weight[expert_indices, ...], self.mlp2_bias[expert_indices, ...]

        t = torch.einsum('sei,seih->seh', t, expert_mlp2_weight) + expert_mlp2_bias
        
        t = torch.einsum('se,seh->sh', expert_weights, t)
        return t


moe = MoEBlock()
x = torch.randn(S, H) # assume flattened batch index
moe(x)

tensor([[ 21030.2539,  65806.7656,  62543.1797,  ..., -63999.4648,
          19397.2969,  -2325.2461],
        [-21176.1152, -24574.4062,  24010.8516,  ...,  15217.9492,
          92997.5469, -25120.8828],
        [-69550.8906,  75468.5547, -27889.4570,  ...,  34576.7305,
          -1586.8516,  -4206.1602],
        ...,
        [ 23120.8164,  54874.9609, -48149.4883,  ...,  19494.6992,
         -39043.8945,  44254.5781],
        [  6686.0176, 106855.1562,   4450.0176,  ...,   6314.5684,
         -43753.0938,  41508.1484],
        [-79605.2344,  79492.6016, -29231.5273,  ..., -17302.1426,
         -45036.5078, -31591.8535]], grad_fn=<ViewBackward0>)

In [18]:
# --- roofline ---

S, H, E, I = 1, 2880, 4, 2880 # flattened seq, hidden size, topk experts, intermediate size

bytes_per_elem = 2 # bf16
flop, mem = 0, 0

# Layernorm: omitted

# Routing: X @ W_gate => (S, H) @ (H, E) = (S, E) => S * E * 2H
flop += S * E * 2 * H
mem += (S * H + H * E + S * E) * bytes_per_elem

# MLP1: X @ W_mlp1 => (S, H) @ (E, S, H, 2I) = (E, S, 2I) => E * S * 2I * 2H
flop += E * S * 2 * I * 2 * H
mem += (S * H + E * S * H * 2 * I + E * S * 2 * I) * bytes_per_elem

# SwiGLU: omitted

# MLP2: X @ W_mlp2 => (E, S, I) @ (E, S, I, H) = (E, S, H) => E * S * H * 2I
flop += E * S * H * 2 * I
mem += (E * S * I + E * S * I * H + E * S * H) * bytes_per_elem

# Expert weighting: X @ W_experts => (E, S, H) @ (S, E) = (S, H) => S * H * 2E
flop += S * H * 2 * E
mem += (E * S * H + S * E + S * H) * bytes_per_elem

print(flop / 1e9, 'GFLOP', mem / 1e9, 'GB')
print(flop / mem, 'FLOP/Byte') # < 428 (H100 FLOP/byte threshold). So memory bound.

0.19911168 GFLOP 0.199221136 GB
0.9994505803842018 FLOP/Byte
